## Vacuum Cleaner Simulation

This simulation demonstrates two types of vacuum cleaner agents: a **Simple Reflex Agent** and a **Goal-Based Agent**.

-   **Simple Reflex Agent**: Decides its actions based *only* on the current percept (e.g., if the current room is dirty, clean it). It doesn't maintain any internal state or memory of past states.
-   **Goal-Based Agent**: Decides its actions based on the current percept, its internal state (which rooms it has already visited and their status), and its overall goal (to clean all rooms).

In [1]:
import random

class Room:
    def __init__(self, room_id, is_dirty=False):
        self.room_id = room_id
        self.is_dirty = is_dirty

    def clean(self):
        self.is_dirty = False

    def __str__(self):
        return f"Room {self.room_id}: {'Dirty' if self.is_dirty else 'Clean'}"

class Environment:
    def __init__(self, num_rooms):
        self.rooms = [Room(i, random.choice([True, False])) for i in range(num_rooms)]
        self.current_location = 0

    def get_room_status(self, room_id):
        return self.rooms[room_id].is_dirty

    def clean_room(self, room_id):
        self.rooms[room_id].clean()

    def all_rooms_clean(self):
        return all(not room.is_dirty for room in self.rooms)

    def display_status(self):
        print("Current Environment Status:")
        for room in self.rooms:
            print(f"  {room}")
        print(f"  Vacuum cleaner is in Room {self.current_location}")


In [5]:
class VacuumCleanerAgent:
    def __init__(self, environment):
        self.environment = environment
        self.actions_taken = 0

    def perceive(self):
        return self.environment.get_room_status(self.environment.current_location)

    def act(self):
        raise NotImplementedError("Subclasses must implement the 'act' method")

    def move_to_next_room(self):
        self.environment.current_location = (self.environment.current_location + 1) % len(self.environment.rooms)
        self.actions_taken += 1

class SimpleReflexVacuumCleaner(VacuumCleanerAgent):
    def __init__(self, environment):
        super().__init__(environment)
        print("Simple Reflex Agent created.")

    def act(self):
        current_room_is_dirty = self.perceive()
        current_room_id = self.environment.current_location

        if current_room_is_dirty:
            print(f"  Agent in Room {current_room_id}: DIRTY. Cleaning...")
            self.environment.clean_room(current_room_id)
            self.actions_taken += 1
            return "Cleaned"
        else:
            print(f"  Agent in Room {current_room_id}: CLEAN. Moving to next room.")
            self.move_to_next_room()
            return "Moved"


class GoalBasedVacuumCleaner(VacuumCleanerAgent):
    def __init__(self, environment):
        super().__init__(environment)
        self.known_room_states = {room.room_id: room.is_dirty for room in environment.rooms}
        print("Goal-Based Agent created.")

    def perceive(self):
        current_room_id = self.environment.current_location
        current_room_is_dirty = self.environment.get_room_status(current_room_id)
        self.known_room_states[current_room_id] = current_room_is_dirty
        return current_room_is_dirty

    def act(self):
        current_room_id = self.environment.current_location
        current_room_is_dirty = self.perceive()

        if current_room_is_dirty:
            print(f"  Agent in Room {current_room_id}: DIRTY. Cleaning...")
            self.environment.clean_room(current_room_id)
            self.known_room_states[current_room_id] = False
            return "Cleaned"
        else:
            print(f"  Agent in Room {current_room_id}: CLEAN.")
            if all(not is_dirty for is_dirty in self.known_room_states.values()):
                print("  Goal achieved: All known rooms are clean. Stopping.")
                return "Stop"
            else:
                next_dirty_room_id = -1
                for i in range(len(self.environment.rooms)):
                    room_to_check = (current_room_id + 1 + i) % len(self.environment.rooms)
                    if self.known_room_states.get(room_to_check, True):
                        next_dirty_room_id = room_to_check
                        break

                if next_dirty_room_id != -1:
                    print(f"  Moving to Room {next_dirty_room_id} (known dirty).")
                    self.environment.current_location = next_dirty_room_id
                    self.actions_taken += 1
                    return "Moved"
                else:
                    print("  No more known dirty rooms. Moving to next available room.")
                    self.move_to_next_room()
                    return "Moved"


In [4]:
def simulate_vacuum_cleaner(agent_type, num_rooms, max_steps=100):
    print(f"\n--- Simulating {agent_type.__name__} with {num_rooms} rooms ---")
    environment = Environment(num_rooms)
    environment.display_status()

    agent = agent_type(environment)

    steps = 0
    while steps < max_steps and not environment.all_rooms_clean():
        print(f"\nStep {steps + 1}:")
        action_result = agent.act()
        if action_result == "Stop":
            break
        environment.display_status()
        steps += 1

    print(f"\nSimulation Finished after {agent.actions_taken} actions.")
    environment.display_status()
    if environment.all_rooms_clean():
        print("All rooms are clean! Goal achieved.")
    else:
        print("Max steps reached or not all rooms clean.")
    print("--------------------------------------------------")


### Run Simulations

In [6]:
simulate_vacuum_cleaner(SimpleReflexVacuumCleaner, num_rooms=3, max_steps=20)

simulate_vacuum_cleaner(GoalBasedVacuumCleaner, num_rooms=3, max_steps=20)



--- Simulating SimpleReflexVacuumCleaner with 3 rooms ---
Current Environment Status:
  Room 0: Dirty
  Room 1: Clean
  Room 2: Clean
  Vacuum cleaner is in Room 0
Simple Reflex Agent created.

Step 1:
  Agent in Room 0: DIRTY. Cleaning...
Current Environment Status:
  Room 0: Clean
  Room 1: Clean
  Room 2: Clean
  Vacuum cleaner is in Room 0

Simulation Finished after 1 actions.
Current Environment Status:
  Room 0: Clean
  Room 1: Clean
  Room 2: Clean
  Vacuum cleaner is in Room 0
All rooms are clean! Goal achieved.
--------------------------------------------------

--- Simulating GoalBasedVacuumCleaner with 3 rooms ---
Current Environment Status:
  Room 0: Clean
  Room 1: Clean
  Room 2: Clean
  Vacuum cleaner is in Room 0
Goal-Based Agent created.

Simulation Finished after 0 actions.
Current Environment Status:
  Room 0: Clean
  Room 1: Clean
  Room 2: Clean
  Vacuum cleaner is in Room 0
All rooms are clean! Goal achieved.
--------------------------------------------------
